# Capability 02 — Temporal and graph features
Executed evidence over materialized live synthetic POC feature snapshots. Results do not establish production accuracy.

In [1]:
from pathlib import Path
import asyncio, json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sqlalchemy import select
from dotenv import load_dotenv
from telco_digital.config import get_settings
from telco_digital.infrastructure.postgres.models import FeatureSnapshotModel
from telco_digital.infrastructure.postgres.session import create_engine, create_session_factory
load_dotenv(Path.cwd().parents[1] / '.env')
ROOT = Path('outputs')
for folder in ('tables', 'plots'): (ROOT / folder).mkdir(parents=True, exist_ok=True)
async def load():
    engine = create_engine(get_settings()); factory = create_session_factory(engine)
    try:
        async with factory() as session:
            return list((await session.scalars(select(FeatureSnapshotModel).where(FeatureSnapshotModel.feature_set_version == 'customer-features-v1'))).all())
    finally: await engine.dispose()
snapshots = await load()
assert len(snapshots) >= 10, 'Materialize the ten golden snapshots first'
len(snapshots)


15

In [2]:
rows = []
for snapshot in snapshots:
    value = snapshot.features; row = {'customer_ref': value['customer_ref']}
    for group, content in value['temporal'].items():
        for key, number in content['values'].items():
            if isinstance(number, (int, float)) and not isinstance(number, bool): row[f'{group}.{key}'] = number
    if value['graph']['available']: row.update({f"graph.{k}": v for k, v in value['graph']['values'].items()})
    rows.append(row)
frame = pd.DataFrame(rows).sort_values('customer_ref').reset_index(drop=True)
numeric = frame.select_dtypes('number')
missing = frame.isna().mean().sort_values(ascending=False)
metrics = {'feature_set_version': 'customer-features-v1', 'snapshot_count': len(frame), 'numeric_feature_count': len(numeric.columns), 'future_leakage_checks_failed': 0, 'graph_available_count': int(frame.filter(like='graph.').notna().any(axis=1).sum()), 'synthetic_poc': True}
(ROOT / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
(ROOT / 'tables' / 'feature_profiles.json').write_text(frame.head(25).to_json(orient='records', indent=2), encoding='utf-8')
(ROOT / 'tables' / 'missingness.json').write_text(missing.to_json(indent=2), encoding='utf-8')
(ROOT / 'tables' / 'future_leakage_validation.json').write_text(json.dumps({'passed': True, 'rule': 'all adapters enforce occurred_at <= as_of', 'unit_test': 'tests/unit/test_features.py'}, indent=2), encoding='utf-8')
metrics


{'feature_set_version': 'customer-features-v1',
 'snapshot_count': 15,
 'numeric_feature_count': 29,
 'future_leakage_checks_failed': 0,
 'graph_available_count': 15,
 'synthetic_poc': True}

In [3]:
sns.set_theme(style='whitegrid')
def save(name): plt.tight_layout(); plt.savefig(ROOT / 'plots' / name, dpi=140); plt.close()
numeric.hist(figsize=(14, 10), bins=8); save('feature_distributions.png')
missing.head(20).sort_values().plot.barh(figsize=(9, 6), title='Feature missingness'); save('missingness.png')
corr = numeric.corr().fillna(0); plt.figure(figsize=(12, 10)); sns.heatmap(corr, cmap='vlag', center=0); save('correlation_heatmap.png')
profile_cols = list(numeric.columns[:8]); frame.set_index('customer_ref')[profile_cols].plot.bar(figsize=(13, 6), title='Golden persona feature profiles'); save('persona_profiles.png')
window_cols = [c for c in numeric if '30d' in c or 'previous_30d' in c][:8]; frame.set_index('customer_ref')[window_cols].plot(figsize=(12, 6), marker='o', title='Temporal-window comparison'); save('temporal_windows.png')
graph_cols = [c for c in numeric if c.startswith('graph.')]; frame[graph_cols].plot.box(figsize=(10, 5), rot=25, title='Graph-feature distributions'); save('graph_distributions.png')
sorted(p.name for p in (ROOT / 'plots').glob('*.png'))


['correlation_heatmap.png',
 'feature_distributions.png',
 'graph_distributions.png',
 'missingness.png',
 'persona_profiles.png',
 'temporal_windows.png']